In [ ]:
# data presentation functions
import matplotlib.pyplot as plt
import snowflake_functions as sf 
import pandas as pd
import numpy as np

# make a histogram of the most often asked for skills.
def hist_popular_skills(connection, hist_file_name):
    # Exclude certain skills
    exclusions = ["R", "Data Engineering"]

    # Get tables from snowflake
    jobs_df = pd.read_sql("SELECT job_id FROM jobs", connection)  
    job_skills_df = pd.read_sql("SELECT skill_id FROM job_skills", connection)
    skills_df = pd.read_sql("SELECT skill_id, skill_name FROM skills_list", connection)

    # Convert skill_id to the same type (string) in both DataFrames
    job_skills_df["skill_id"] = job_skills_df["skill_id"].astype(str)
    skills_df["skill_id"] = skills_df["skill_id"].astype(str)
    
    # Merge to get skill names
    merged_df = job_skills_df.merge(skills_df, on="skill_id", how="left")
    
    # Count skill occurrences
    skill_counts = merged_df["skill_name"].value_counts()

    # Filter out exclusions from the skill counts
    skill_counts = skill_counts[~skill_counts.index.isin(exclusions)]
    
    # Compute percentage of total job postings
    total_jobs = len(jobs_df)
    skill_percentages = (skill_counts / total_jobs) * 100
    
    # Plot the histogram
    plt.figure(figsize=(12, 6))
    skill_percentages.head(20).plot(kind="bar")  # Show top 20 skills
    plt.xlabel("Skill Name")
    plt.ylabel("Percentage of Job Listings (%)")
    plt.title("Most Requested Skills as Percentage of Job Listings")
    plt.xticks(rotation=45, ha="right")
    plt.show()

    # Save the figure to the file
    plt.savefig(hist_file_name, bbox_inches='tight')  # Save as PNG, adjust for tight layout
    plt.close()


# Function to plot annual salaries from jobs table.
def hist_annual_salaries(connection, min_salary, max_salary, salary_file_name):
    # Get tables from Snowflake
    jobs_df, job_skills_df, skills_df = sf.get_snowflake_tables(connection)

    # Ensure salary column has valid values before filtering
    jobs_df = jobs_df[jobs_df['salary'].notna()].copy()  # Use .copy() to avoid chained assignment warnings
    jobs_df.loc[:, 'salary'] = pd.to_numeric(jobs_df['salary'], errors='coerce')  # Ensure numeric

    # Filter jobs with pay period 'annually' and employment_type 'Full time'
    annual_jobs_df = jobs_df.loc[
        (jobs_df['pay_period'] == 'annually') & 
        (jobs_df['employment_type'] == 'Full time') & 
        (jobs_df['salary'] >= min_salary) & 
        (jobs_df['salary'] <= max_salary)
    ].copy()  # Use .copy() to avoid the warning

    # Convert salary to $1000 units
    annual_jobs_df.loc[:, "salary"] = annual_jobs_df["salary"] / 1_000  

    # Define bin edges in units of $1000
    bin_edges = np.arange(min_salary / 1_000, (max_salary / 1_000) + 10, 10)

    # Plot the histogram
    plt.figure(figsize=(12, 6))
    plt.hist(annual_jobs_df["salary"], bins=bin_edges, edgecolor="black", color="royalblue")
    plt.xlabel("Annual Salary ($1000)")
    plt.ylabel("Number of Jobs")
    plt.title("Distribution of Annual Salaries")
    
    # Set x-ticks directly in thousands (e.g., 10, 20, ..., 200)
    plt.xticks(bin_edges, labels=[f"{int(b)}" for b in bin_edges], rotation=45)
    
    plt.grid(axis="y", linestyle="--", alpha=0.7)  # Improves readability
    plt.show()
    
    # Save the figure to the file
    plt.savefig(salary_file_name, bbox_inches='tight')  # Save as PNG, adjust for tight layout
    plt.close()